# Matchbox FHIR → OMOP Demo
Covers server health checks, Echidna terminology lookups, and FHIR resource transforms.

In [ ]:
import os
import sys
import json
import requests
import pandas as pd
from IPython.display import display

SCRIPTS_DIR = '/home/jovyan/matchbox_scripts'
ECHIDNA_URL = 'https://echidna.fhir.org/r4'
BASE_URL = os.environ.get('MATCHBOX_URL', 'http://matchbox:8080') + '/matchboxv3/fhir'
HEADERS_JSON = {'Accept': 'application/fhir+json'}

# transforms.py is the shared ETL module — also used by load_duckdb.py
sys.path.insert(0, SCRIPTS_DIR)
from transforms import (
    transform_condition, transform_patient, transform_procedure,
    transform_allergy, transform_encounter, transform_immunization,
    transform_measurement, transform_observation, transform_vital_signs,
    transform_medication,
)

print(f'Matchbox: {BASE_URL}')

## 1. Server Health Checks

In [22]:
# Verify OMOP IG $transform maps are loaded
r = requests.get(f'{BASE_URL}/StructureMap', headers=HEADERS_JSON)
bundle = r.json()

omop_maps = [
    {'name': e['resource'].get('name'), 'url': e['resource'].get('url')}
    for e in bundle.get('entry', [])
    if (e['resource'].get('url') or '').startswith('http://hl7.org/fhir/uv/omop/')
]

if omop_maps:
    print(f"OMOP StructureMaps loaded ({len(omop_maps)}) — $transform is ready:")
    display(pd.DataFrame(omop_maps))
else:
    print('WARNING: No OMOP StructureMaps found — IG may not be loaded')

OMOP StructureMaps loaded (11) — $transform is ready:


,name,url
0,ProcedureMap,http://hl7.org/fhir/uv/omop/StructureMap/Proce...
1,ConditionMap,http://hl7.org/fhir/uv/omop/StructureMap/Condi...
2,ImmunizationMap,http://hl7.org/fhir/uv/omop/StructureMap/Immun...
3,EncounterVisitMap,http://hl7.org/fhir/uv/omop/StructureMap/Encou...
4,SimpleVitalSignsMap,http://hl7.org/fhir/uv/omop/StructureMap/Simpl...
5,PersonMap,http://hl7.org/fhir/uv/omop/StructureMap/Perso...
6,BloodPressureVitalSignsMap,http://hl7.org/fhir/uv/omop/StructureMap/Blood...
7,MeasurementMap,http://hl7.org/fhir/uv/omop/StructureMap/Measu...
8,ObservationMap,http://hl7.org/fhir/uv/omop/StructureMap/Obser...
9,AllergyMap,http://hl7.org/fhir/uv/omop/StructureMap/Aller...


In [23]:
# Check OMOP IG is loaded (check_ig_loaded.sh)
r = requests.get(f'{BASE_URL}/ImplementationGuide', params={'_content': 'omop'}, headers=HEADERS_JSON)
bundle = r.json()
total = bundle.get('total', 0)
print(f"OMOP IGs found: {total}")
for entry in bundle.get('entry', []):
    ig = entry['resource']
    print(f"  {ig.get('name')} {ig.get('version')}")

OMOP IGs found: 2
  None 1.0.0
  None 7.1.0


## 2. Terminology Lookups via Echidna

In [24]:
def lookup_snomed(code, label=''):
    r = requests.get(
        f'{ECHIDNA_URL}/ConceptMap/$translate',
        headers=HEADERS_JSON,
        params={
            'system': 'http://snomed.info/sct',
            'code': code,
            'targetsystem': 'https://fhir-terminology.ohdsi.org',
        }
    )
    result = r.json()
    matched = next((p for p in result.get('parameter', []) if p['name'] == 'result'), {})
    concepts = [p for p in result.get('parameter', []) if p['name'] == 'match']
    print(f"[{label or code}] matched={matched.get('valueBoolean')}")
    for m in concepts:
        for part in m.get('part', []):
            if part['name'] == 'concept':
                c = part['valueCoding']
                print(f"  OMOP concept_id={c.get('code')}  display={c.get('display')}")
    return result

In [25]:
# lookup_hypertension.sh
lookup_snomed('38341003', 'Hypertensive disorder');

[Hypertensive disorder] matched=True
  OMOP concept_id=316866  display=None


In [26]:
# lookup_fever.sh
lookup_snomed('386661006', 'Fever');

[Fever] matched=True
  OMOP concept_id=437663  display=None


In [27]:
# lookup_unknown.sh — bogus code, expect no match
lookup_snomed('0000001', 'Unknown/bogus code');

[Unknown/bogus code] matched=False


## 3. FHIR → OMOP Transforms

In [ ]:
OMOP_COLUMNS = {
    'ConditionOccurrence': [
        'condition_occurrence_id', 'person_id', 'condition_concept_id',
        'condition_start_date', 'condition_start_datetime', 'condition_end_date',
        'condition_end_datetime', 'condition_type_concept_id', 'condition_status_concept_id',
        'stop_reason', 'provider_id', 'visit_occurrence_id', 'visit_detail_id',
        'condition_source_value', 'condition_source_concept_id', 'condition_status_source_value',
    ],
    'ProcedureOccurrence': [
        'procedure_occurrence_id', 'person_id', 'procedure_concept_id',
        'procedure_date', 'procedure_datetime', 'procedure_end_date', 'procedure_end_datetime',
        'procedure_type_concept_id', 'modifier_concept_id', 'quantity',
        'provider_id', 'visit_occurrence_id', 'visit_detail_id',
        'procedure_source_value', 'procedure_source_concept_id', 'modifier_source_value',
    ],
    'Person': [
        'person_id', 'gender_concept_id', 'year_of_birth', 'month_of_birth', 'day_of_birth',
        'birth_datetime', 'race_concept_id', 'ethnicity_concept_id', 'location_id',
        'provider_id', 'care_site_id', 'person_source_value', 'gender_source_value',
        'gender_source_concept_id', 'race_source_value', 'race_source_concept_id',
        'ethnicity_source_value', 'ethnicity_source_concept_id',
    ],
    'Observation': [
        'observation_id', 'person_id', 'observation_concept_id', 'observation_date',
        'observation_datetime', 'observation_type_concept_id', 'value_as_number',
        'value_as_string', 'value_as_concept_id', 'qualifier_concept_id', 'unit_concept_id',
        'provider_id', 'visit_occurrence_id', 'visit_detail_id', 'observation_source_value',
        'observation_source_concept_id', 'unit_source_value', 'qualifier_source_value',
    ],
    'Measurement': [
        'measurement_id', 'person_id', 'measurement_concept_id', 'measurement_date',
        'measurement_datetime', 'measurement_type_concept_id', 'operator_concept_id',
        'value_as_number', 'value_as_concept_id', 'unit_concept_id',
        'range_low', 'range_high', 'provider_id', 'visit_occurrence_id', 'visit_detail_id',
        'measurement_source_value', 'measurement_source_concept_id',
        'unit_source_value', 'value_source_value',
    ],
    'DrugExposure': [
        'drug_exposure_id', 'person_id', 'drug_concept_id',
        'drug_exposure_start_date', 'drug_exposure_start_datetime',
        'drug_exposure_end_date', 'drug_exposure_end_datetime',
        'drug_type_concept_id', 'stop_reason', 'refills', 'quantity', 'days_supply',
        'route_concept_id', 'lot_number', 'provider_id', 'visit_occurrence_id', 'visit_detail_id',
        'drug_source_value', 'drug_source_concept_id', 'route_source_value', 'dose_unit_source_value',
    ],
    'VisitOccurrence': [
        'visit_occurrence_id', 'person_id', 'visit_concept_id',
        'visit_start_date', 'visit_start_datetime', 'visit_end_date', 'visit_end_datetime',
        'visit_type_concept_id', 'provider_id', 'care_site_id',
        'visit_source_value', 'visit_source_concept_id',
        'admitted_from_concept_id', 'admitted_from_source_value',
        'discharged_to_concept_id', 'discharged_to_source_value',
    ],
}

def show(transform_fn, resource, label=''):
    """Call a transforms.py function and display the result as a DataFrame."""
    result = transform_fn(resource)
    rtype = (result or {}).get('resourceType', 'suppressed')
    print(f"[{label}] resourceType={rtype}")
    if result is None:
        print('  (suppressed)')
        return None
    cols = OMOP_COLUMNS.get(rtype)
    if cols:
        display(pd.DataFrame([{c: result.get(c, '') for c in cols}]))
    elif rtype == 'Bundle':
        rows = []
        for entry in result.get('entry', []):
            res = entry.get('resource', {})
            ecols = OMOP_COLUMNS.get(res.get('resourceType'))
            if ecols:
                rows.append({c: res.get(c, '') for c in ecols})
        display(pd.DataFrame(rows)) if rows else print(json.dumps(result, indent=2))
    else:
        print(json.dumps(result, indent=2))
    return result

def load(filename):
    with open(f'{SCRIPTS_DIR}/{filename}') as f:
        return json.load(f)

In [ ]:
show(transform_patient, load('patient.json'), 'Patient→Person');

In [ ]:
show(transform_condition, load('condition_hypertension.json'), 'Hypertension');

In [ ]:
show(transform_condition, load('condition_fever.json'), 'Fever');

In [ ]:
# verificationStatus=refuted — transforms.py suppresses this (ConditionMap bug workaround)
show(transform_condition, load('condition_refuted.json'), 'Refuted condition (suppressed by transforms.py)');

In [ ]:
show(transform_condition, load('condition_unknown_code.json'), 'Unknown SNOMED code');

In [ ]:
show(transform_procedure, load('procedure_completed.json'), 'Procedure completed');

In [ ]:
# status=not-done — transforms.py suppresses this (ProcedureMap bug workaround)
show(transform_procedure, load('procedure_not_done.json'), 'Procedure not-done (suppressed by transforms.py)');

## 4. Upload PersonMap (FML)

## 4. Additional FHIR → OMOP Transforms

In [ ]:
show(transform_allergy, load('allergy_peanut.json'), 'Allergy peanut');

In [ ]:
show(transform_encounter, load('encounter_outpatient.json'), 'Encounter outpatient');

In [ ]:
show(transform_immunization, load('immunization_flu.json'), 'Immunization flu');

In [ ]:
# Decimal value (72.5 kg) fails in MeasurementMap with NumberFormatException
show(transform_measurement, load('observation_weight.json'), 'Body weight (decimal - known failure)');

In [ ]:
show(transform_measurement, load('observation_weight_int.json'), 'Body weight (integer)');

In [ ]:
show(transform_observation, load('observation_smoking.json'), 'Smoking status');

In [ ]:
# Decimal value (37.2 Cel) fails in SimpleVitalSignsMap with NumberFormatException
show(transform_vital_signs, load('observation_temperature.json'), 'Body temperature (decimal - known failure)');

In [ ]:
show(transform_vital_signs, load('observation_temperature_int.json'), 'Body temperature (integer)');

In [ ]:
# Known failure: matchbox cannot set measurement_type_concept_id on Bundle output
# Error: HAPI-0389: java.lang.Error: Cannot set property measurement_type_concept_id on resource
show(transform_vital_signs, load('observation_blood_pressure.json'), 'Blood pressure (known failure)');

In [ ]:
show(transform_medication, load('medication_aspirin.json'), 'Aspirin');

In [20]:
# upload_personmap.sh
with open(f'{SCRIPTS_DIR}/PersonMap.fml') as f:
    fml = f.read()

r = requests.post(
    f'{BASE_URL}/StructureMap',
    headers={'Content-Type': 'text/fhir-mapping', 'Accept': 'application/fhir+json'},
    data=fml.encode(),
)
print(f"Status: {r.status_code}")
result = r.json()
print(f"id={result.get('id')}  url={result.get('url')}")

Status: 201
id=PersonMap  url=http://hl7.org/fhir/uv/omop/StructureMap/PersonMap
